In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Milestone - 3**

In [2]:
!pip install faiss-cpu -q

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Creating knowledge base...")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index...")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=True)
kb_embeddings = np.array(kb_embeddings).astype('float32')
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")
print("KB size:", len(kb))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.7 MB/s eta 0:00:00:00:0100:01
Creating knowledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge base successfully created
KB size: 2000


# **Zero-shot classifier for Q1, Q2, Q6**

In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150    = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
              str(row_150['D']), str(row_150['E'])]
ans_150    = str(row_150[row_150['answer']])

print("Correct answer text:", ans_150)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Correct answer text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


Q 1) **Zero-shot probability of correct option**

In [4]:
result_q1 = zs(prompt_150, labels_150)

print("Labels :", result_q1['labels'])
print("Scores :", result_q1['scores'])

# Find score of correct answer
for label, score in zip(result_q1['labels'], result_q1['scores']):
    if label == ans_150:
        print(f"\nAnswer Q1 — Probability of correct option: {round(score, 3)}")
        break

Labels : ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon th

Q 2) **FAISS rank of true document**  

In [5]:
prompt_emb = model.encode([prompt_150]).astype('float32')

k = 10
distances, indices = index.search(prompt_emb, k)
retrieved_indices  = indices[0].tolist()

print("Retrieved indices:", retrieved_indices)
print("True index (150) in retrieved:", 150 in retrieved_indices)

if 150 in retrieved_indices:
    rank = retrieved_indices.index(150) + 1
    print(f"Answer Q2 — FAISS rank of true document: {rank}")
else:
    print("True document NOT found in top 10")

Retrieved indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
True index (150) in retrieved: True
Answer Q2 — FAISS rank of true document: 10


Q 3) **Cross-Encoder reranking rank**

In [6]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
pairs   = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score
sorted_idx = np.argsort(ce_scores)[::-1]
reranked_indices = [retrieved_indices[i] for i in sorted_idx]

print("Reranked indices:", reranked_indices)

if 150 in reranked_indices:
    rank_ce = reranked_indices.index(150) + 1
    print(f"Answer Q3 — Cross-Encoder rank of true document: {rank_ce}")
else:
    print("True document NOT in reranked list")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranked indices: [150, 1906, 847, 1693, 1269, 1532, 168, 576, 1701, 663]
Answer Q3 — Cross-Encoder rank of true document: 1


Q 4) **Token count of RAG string for row 42**

In [7]:
tokenizer_bert = AutoTokenizer.from_pretrained('bert-base-uncased')

row_42     = train.iloc[42]
prompt_42  = str(row_42['prompt'])
prompt_emb_42 = model.encode([prompt_42]).astype('float32')

_, indices_42 = index.search(prompt_emb_42, 5)
docs_5 = [kb[i] for i in indices_42[0]]

concatenated = ' '.join(docs_5)
rag_string   = f"Context: {concatenated} Question: {prompt_42}"

tokens = tokenizer_bert(rag_string, truncation=False)
print(f"RAG string: {rag_string[:200]}...")
print(f"Answer Q4 — Total tokens: {len(tokens['input_ids'])}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

RAG string: Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combinati...
Answer Q4 — Total tokens: 216


Q 5) **RAG with true document, new probability**

In [8]:
true_doc_150 = kb[150]
rag_str_q5   = f"Context: {true_doc_150} Question: {prompt_150}"

result_q5 = zs(rag_str_q5, labels_150)

for label, score in zip(result_q5['labels'], result_q5['scores']):
    if label == ans_150:
        print(f"Answer Q5 — New probability with RAG: {round(score, 3)}")
        break

print("\nAll scores:")
for label, score in zip(result_q5['labels'], result_q5['scores']):
    print(f"  {label}: {round(score, 3)}")

Answer Q5 — New probability with RAG: 0.989

All scores:
  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.989
  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.004
  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.003
  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The co

Q 6) **Adversarial RAG with KB index 999**

In [9]:
adversarial_doc = kb[999]
rag_str_q6      = f"Context: {adversarial_doc} Question: {prompt_150}"

result_q6 = zs(rag_str_q6, labels_150)

for label, score in zip(result_q6['labels'], result_q6['scores']):
    if label == ans_150:
        print(f"Answer Q6 — Adversarial RAG probability: {round(score, 3)}")
        break

print("\nAll scores:")
for label, score in zip(result_q6['labels'], result_q6['scores']):
    print(f"  {label}: {round(score, 3)}")

Answer Q6 — Adversarial RAG probability: 0.529

All scores:
  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.529
  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.425
  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.02
  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos.": 0.019
  The butterfly effect is the phenomenon that a sma

Q 7) **Hit Rate for first 100 rows**

In [10]:
hits = 0

for i in range(100):
    row        = train.iloc[i]
    prompt_i   = str(row['prompt'])
    correct_text = str(row[row['answer']])

    prompt_emb_i   = model.encode([prompt_i]).astype('float32')
    _, indices_i   = index.search(prompt_emb_i, 5)
    retrieved_docs = [kb[j] for j in indices_i[0]]

    for doc in retrieved_docs:
        if correct_text in doc:
            hits += 1
            break

hit_rate = (hits / 100) * 100
print(f"Hits        : {hits}/100")
print(f"Answer Q7 — Hit Rate: {round(hit_rate, 1)}%")

Hits        : 73/100
Answer Q7 — Hit Rate: 73.0%


Q 8) **Full RAG pipeline MAP@3 for first 20 rows**

In [11]:
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

scores_q8 = []

for i in range(20):
    row       = train.iloc[i]
    prompt_i  = str(row['prompt'])
    answer_i  = row['answer']
    labels_i  = [str(row[c]) for c in OPTION_COLS]

    # Step 1 — Retrieve top 5
    prompt_emb_i = model.encode([prompt_i]).astype('float32')
    _, indices_i = index.search(prompt_emb_i, 5)
    docs_5_i     = [kb[j] for j in indices_i[0]]

    # Step 2 — Rerank with Cross-Encoder
    pairs_i    = [[prompt_i, doc] for doc in docs_5_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc   = docs_5_i[np.argmax(ce_scores_i)]

    # Step 3 — Augment
    rag_str_i = f"Context: {best_doc} Question: {prompt_i}"

    # Step 4 — Predict
    result_i = zs(rag_str_i, labels_i)

    # Step 5 — Score
    label_score_pairs = list(zip(result_i['labels'], result_i['scores']))
    label_score_pairs = sorted(label_score_pairs, key=lambda x: x[1], reverse=True)

    top3_letters = []
    for label, score in label_score_pairs[:3]:
        idx_opt = labels_i.index(label)
        top3_letters.append(OPTION_COLS[idx_opt])

    row_map3 = map_at_3(answer_i, top3_letters)
    scores_q8.append(row_map3)

    print(f"Row {i} | Answer: {answer_i} | Top3: {top3_letters} | MAP@3: {row_map3:.3f}")

final_map3 = np.mean(scores_q8)
print(f"\nAnswer Q8 — Final RAG Pipeline MAP@3: {round(final_map3, 3)}")

Row 0 | Answer: B | Top3: ['B', 'D', 'A'] | MAP@3: 1.000
Row 1 | Answer: A | Top3: ['A', 'E', 'C'] | MAP@3: 1.000
Row 2 | Answer: C | Top3: ['C', 'D', 'B'] | MAP@3: 1.000
Row 3 | Answer: B | Top3: ['B', 'D', 'A'] | MAP@3: 1.000
Row 4 | Answer: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.000
Row 5 | Answer: C | Top3: ['B', 'C', 'A'] | MAP@3: 0.500
Row 6 | Answer: E | Top3: ['E', 'B', 'D'] | MAP@3: 1.000
Row 7 | Answer: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.000
Row 8 | Answer: A | Top3: ['A', 'C', 'D'] | MAP@3: 1.000
Row 9 | Answer: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.000
Row 10 | Answer: C | Top3: ['C', 'A', 'D'] | MAP@3: 1.000
Row 11 | Answer: B | Top3: ['B', 'C', 'A'] | MAP@3: 1.000
Row 12 | Answer: D | Top3: ['D', 'A', 'B'] | MAP@3: 1.000
Row 13 | Answer: E | Top3: ['E', 'D', 'B'] | MAP@3: 1.000
Row 14 | Answer: E | Top3: ['E', 'A', 'D'] | MAP@3: 1.000
Row 15 | Answer: E | Top3: ['E', 'B', 'C'] | MAP@3: 1.000
Row 16 | Answer: C | Top3: ['C', 'B', 'E'] | MAP@3: 1.000
Row 17 | Answer: C | Top